In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE EmbedBwdFused BITEXACT TEST (PURE TKV, no calloc) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so + .tkv C .so. NO C injection. CUNG flags.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/embed_bwd_pure.cpu.c'; so=ex+'/embed_bwd_pure.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_embed_bwd' in txt, '(expect False)')
print('no calloc =', 'calloc' not in txt)
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare EmbedBwdFused
import ctypes, numpy as np
PP=ctypes.POINTER(ctypes.c_float); PI=ctypes.POINTER(ctypes.c_longlong); II=ctypes.c_int
args=[PI,II,II,II,PP,II,PP,II]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').EmbedBwdFused
native.argtypes=args; native.restype=II
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/embed_bwd_pure.so').EmbedBwdFused
tkv.argtypes=args; tkv.restype=II
rng=np.random.default_rng(47)
for rows,d,n in [(16,8,32),(64,16,100),(128,32,500),(7,8,20)]:
    idx=rng.integers(0,rows,n).astype(np.int64)
    gy=rng.standard_normal(n*d).astype(np.float32)
    ga=np.zeros(rows*d,np.float32); gc=np.zeros(rows*d,np.float32)
    ia=idx.ctypes.data_as(PI); ya=gy.ctypes.data_as(PP)
    native(ia,n,rows,d,ya,gy.size,ga.ctypes.data_as(PP),ga.size)
    tkv  (ia,n,rows,d,ya,gy.size,gc.ctypes.data_as(PP),gc.size)
    diff=np.count_nonzero(ga.view(np.uint32)!=gc.view(np.uint32))
    print('rows=%d d=%d n=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(rows,d,n,diff==0,diff,ga.size,np.abs(ga-gc).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> EmbedBwdFused pure .tkv (direct scatter) replaces auto-glue C.')